# Resource Estimation

Resource estimation is a first-class feature of qdk-pythonic. This notebook shows how to estimate
the physical resources needed for a quantum circuit, and how to compare different configurations.

In [ ]:
from qdk_pythonic import Circuit

## Build a 3-Qubit GHZ State

The GHZ state generalizes the Bell state to 3+ qubits: `|000> + |111>` (unnormalized).

In [ ]:
circ = Circuit()
q = circ.allocate(3, label="ghz")

circ.h(q[0]).cx(q[0], q[1]).cx(q[1], q[2])

print(circ.draw())
print(f"Gate count: {circ.gate_count()}, Depth: {circ.depth()}")

## Basic Resource Estimation

> **Note:** All estimation cells below require `qsharp>=1.25` (`pip install qsharp>=1.25`).

Call `estimate()` with no arguments to use the default qubit model and error budget.

In [ ]:
# Requires: pip install qsharp>=1.25
result = circ.estimate()
print(result)

## Custom Parameters

Pass a dictionary to `estimate()` to configure the qubit model, error budget, and other settings.
These parameters are forwarded directly to `qsharp.estimate()`.

In [ ]:
# Requires: pip install qsharp>=1.25
params = {
    "qubitParams": {"name": "qubit_maj_ns_e6"},
    "errorBudget": 0.01,
}

result = circ.estimate(params=params)
print(result)

## Batch Estimation

Use `estimate_circuit_batch` to compare multiple circuits or parameter configurations in one call.
This is useful for exploring how different qubit models or error budgets affect resource costs.

In [ ]:
# Requires: pip install qsharp>=1.25
from qdk_pythonic.execution import estimate_circuit_batch

# Build two circuits of different sizes
ghz_3 = Circuit()
q3 = ghz_3.allocate(3)
ghz_3.h(q3[0]).cx(q3[0], q3[1]).cx(q3[1], q3[2])

ghz_5 = Circuit()
q5 = ghz_5.allocate(5)
ghz_5.h(q5[0])
for i in range(4):
    ghz_5.cx(q5[i], q5[i + 1])

# Compare both circuits with two qubit models
configs = [
    {"qubitParams": {"name": "qubit_gate_ns_e3"}},
    {"qubitParams": {"name": "qubit_maj_ns_e6"}},
]

results = estimate_circuit_batch([ghz_3, ghz_5], configs)
for r in results:
    print(r)
    print()